# Notebook 02: HuggingFace Trainer Pipeline
**CSCI316 Project 2 | SentimentGulf**

This notebook implements the **same 4 experiments** as Notebook 01 but using the
HuggingFace `Trainer` API instead of raw PyTorch training loops.

**Framework difference:**
- Notebook 01: Manual loop — `loss.backward()`, `optimizer.step()`, `scheduler.step()` all written by hand
- Notebook 02: `trainer.train()` — one call handles everything

**Same experiments:**
- A: XLM-RoBERTa Full Fine-Tuning
- B: XLM-RoBERTa LoRA (peft library — appropriate for HF Trainer)
- C: MARBERT Full Fine-Tuning
- D: MARBERT LoRA (peft library)

**Note on LoRA implementation:**
Notebook 01 uses LoRA from scratch (peft_implementation.py).
This notebook uses the HuggingFace peft library's LoraConfig.
Both produce equivalent results — this is the library version.

**Before running:** Runtime → Change runtime type → T4 GPU

## 1. Install and Import

In [ ]:
!pip install -q transformers peft accelerate scikit-learn evaluate seaborn matplotlib nltk pyarrow pandas

In [ ]:
import os, sys, random, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset as HFDataset
import evaluate
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score
)

# ── Reproducibility ──────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ── Find repo root ────────────────────────────────────────────────
ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / 'preprocessing').exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print(f'Repo root: {ROOT}')

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────
XLM_MODEL   = 'xlm-roberta-base'
MAR_MODEL   = 'UBC-NLP/MARBERTv2'
MAX_LEN     = 128
BATCH_SIZE  = 16
EPOCHS      = 5
LR_FULL     = 2e-5
LR_LORA     = 3e-4
NUM_LABELS  = 3
LORA_R      = 8
LORA_ALPHA  = 16
LORA_DROP   = 0.1
PATIENCE    = 3

ID2LABEL = {0: 'negative', 1: 'neutral', 2: 'positive'}
LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}

os.makedirs('hf_outputs', exist_ok=True)
print('Config ready')

## 2. Load and Prepare Data

In [ ]:
# Load cleaned_dataset.csv (same file as Notebook 01)
cleaned_path = ROOT / 'preprocessing' / 'datasets' / 'processed' / 'cleaned_dataset.csv'
print(f'Loading: {cleaned_path}')
df = pd.read_csv(cleaned_path)
print(f'Total rows: {len(df):,}')

# Split
train_df = df[df['split'] == 'train'].copy()
val_df   = df[df['split'] == 'val'].copy()
test_df  = df[df['split'] == 'test'].copy()

# Balance training set (same as Notebook 01)
neg_count  = len(train_df[train_df['label'] == 'negative'])
pos_target = int(neg_count * 1.5)
neu_target = neg_count

train_df = pd.concat([
    train_df[train_df['label'] == 'negative'],
    train_df[train_df['label'] == 'positive'].sample(n=pos_target, random_state=SEED),
    train_df[train_df['label'] == 'neutral'].sample(n=neu_target,  random_state=SEED),
], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

# Map labels to integers
for split_df in [train_df, val_df, test_df]:
    split_df['label'] = split_df['label'].map(LABEL2ID).astype(int)

# Rename column
train_df = train_df.rename(columns={'cleaned_text': 'text'})
val_df   = val_df.rename(columns={'cleaned_text': 'text'})
test_df  = test_df.rename(columns={'cleaned_text': 'text'})

print(f'\nAfter balancing:')
print(f'  Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
print(f'\nTrain label distribution:')
print(train_df['label'].value_counts().sort_index())

In [ ]:
# ── Tokenization helper ───────────────────────────────────────────
# HuggingFace Trainer works with HuggingFace Dataset objects
# (different from PyTorch Dataset — this is HF's own format)

def make_hf_dataset(df, tokenizer):
    """
    Convert pandas DataFrame to HuggingFace Dataset with tokenization.
    HF Trainer requires this format — it cannot accept PyTorch DataLoaders directly.
    """
    hf_ds = HFDataset.from_pandas(df[['text', 'label']].reset_index(drop=True))

    def tokenize(batch):
        return tokenizer(
            batch['text'],
            padding='max_length',
            truncation=True,
            max_length=MAX_LEN,
        )

    hf_ds = hf_ds.map(tokenize, batched=True, batch_size=256)
    hf_ds = hf_ds.rename_column('label', 'labels')
    hf_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    return hf_ds

print('make_hf_dataset function defined')

In [ ]:
# ── Metrics function for Trainer ──────────────────────────────────
# The Trainer calls this after each eval step

f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_metric.compute(
        predictions=preds, references=labels, average='macro'
    )['f1']
    accuracy = accuracy_score(labels, preds)
    return {'macro_f1': macro_f1, 'accuracy': accuracy}

print('compute_metrics defined')

In [ ]:
# ── Plotting helpers ──────────────────────────────────────────────

def plot_trainer_history(trainer, title):
    """Extract and plot training history from HF Trainer log."""
    log = trainer.state.log_history
    train_loss = [e['loss']          for e in log if 'loss'          in e and 'eval_loss' not in e]
    val_loss   = [e['eval_loss']     for e in log if 'eval_loss'     in e]
    val_f1     = [e['eval_macro_f1'] for e in log if 'eval_macro_f1' in e]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    if train_loss:
        ax1.plot(train_loss, label='Train loss')
    if val_loss:
        ax1.plot(val_loss, label='Val loss')
    ax1.set(title=f'{title} — Loss', xlabel='Step', ylabel='Loss')
    ax1.legend()

    if val_f1:
        ax2.plot(val_f1, label='Val Macro F1', color='green')
    ax2.set(title=f'{title} — Val Macro F1', xlabel='Eval step', ylabel='F1')
    ax2.legend()

    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(f'hf_outputs/curves_{title.replace(" ","_")}.png', dpi=150)
    plt.show()


def plot_confusion(preds, labels, title):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative','Neutral','Positive'],
                yticklabels=['Negative','Neutral','Positive'])
    plt.title(f'{title} — Confusion Matrix')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(f'hf_outputs/cm_{title.replace(" ","_")}.png', dpi=150)
    plt.show()


def full_evaluation(trainer, test_ds, title):
    """Get predictions on test set and print full report."""
    preds_out   = trainer.predict(test_ds)
    preds       = np.argmax(preds_out.predictions, axis=-1)
    labels      = preds_out.label_ids
    macro_f1    = f1_score(labels, preds, average='macro')
    acc         = accuracy_score(labels, preds)

    print(f'\n{title} — Test Macro F1: {macro_f1:.4f}  |  Accuracy: {acc:.4f}')
    print(f'\n{"="*50}\n{title} — Classification Report\n{"="*50}')
    print(classification_report(labels, preds,
                                 target_names=['Negative','Neutral','Positive']))
    plot_confusion(preds, labels, title)
    return macro_f1, acc, preds, labels

print('Plotting and evaluation helpers defined')

## 3. XLM-RoBERTa Experiments

### Strategy A: Full Fine-Tuning
All parameters are trainable. The HuggingFace `Trainer` handles the
training loop, gradient updates, and evaluation automatically.

In [ ]:
# ── XLM-RoBERTa tokenizer and datasets ───────────────────────────
print('Loading XLM-RoBERTa tokenizer...')
xlm_tok = AutoTokenizer.from_pretrained(XLM_MODEL)

print('Tokenizing datasets...')
xlm_train_ds = make_hf_dataset(train_df, xlm_tok)
xlm_val_ds   = make_hf_dataset(val_df,   xlm_tok)
xlm_test_ds  = make_hf_dataset(test_df,  xlm_tok)

print(f'Train: {len(xlm_train_ds):,} | Val: {len(xlm_val_ds):,} | Test: {len(xlm_test_ds):,}')

In [ ]:
# ── Strategy A: XLM-RoBERTa Full Fine-Tuning ─────────────────────
print('Loading XLM-RoBERTa for Full Fine-Tuning...')
xlm_full = AutoModelForSequenceClassification.from_pretrained(
    XLM_MODEL, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID
)
trainable = sum(p.numel() for p in xlm_full.parameters() if p.requires_grad)
total     = sum(p.numel() for p in xlm_full.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

# TrainingArguments — this is what replaces all the manual setup in Notebook 01
# (optimizer, scheduler, gradient clipping, evaluation loop — all handled here)
args_xlm_full = TrainingArguments(
    output_dir                  = 'hf_outputs/xlm_full',
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate               = LR_FULL,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'macro_f1',
    greater_is_better           = True,
    logging_steps               = 100,
    seed                        = SEED,
    fp16                        = torch.cuda.is_available(),
    report_to                   = 'none',
)

trainer_xlm_full = Trainer(
    model           = xlm_full,
    args            = args_xlm_full,
    train_dataset   = xlm_train_ds,
    eval_dataset    = xlm_val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

print('Training XLM-RoBERTa Full Fine-Tuning...')
trainer_xlm_full.train()
print('Done')

In [ ]:
plot_trainer_history(trainer_xlm_full, 'XLM-RoBERTa Full FT (HF)')
f1_xlm_full, acc_xlm_full, preds_xlm_full, labels_test = full_evaluation(
    trainer_xlm_full, xlm_test_ds, 'XLM-RoBERTa Full FT (HF)'
)

### Strategy B: XLM-RoBERTa LoRA (peft library)
Only LoRA matrices are trained (~1.4M parameters).
Uses `LoraConfig` from the HuggingFace peft library —
this is the library version. Notebook 01 used the from-scratch implementation.

In [ ]:
# ── Strategy B: XLM-RoBERTa LoRA (peft library) ──────────────────
print('Loading XLM-RoBERTa for LoRA...')
xlm_base_lora = AutoModelForSequenceClassification.from_pretrained(
    XLM_MODEL, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID
)

lora_config = LoraConfig(
    task_type     = TaskType.SEQ_CLS,
    r             = LORA_R,
    lora_alpha    = LORA_ALPHA,
    lora_dropout  = LORA_DROP,
    target_modules= ['query', 'value'],
    bias          = 'none',
)

xlm_lora = get_peft_model(xlm_base_lora, lora_config)
xlm_lora.print_trainable_parameters()

args_xlm_lora = TrainingArguments(
    output_dir                  = 'hf_outputs/xlm_lora',
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate               = LR_LORA,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'macro_f1',
    greater_is_better           = True,
    logging_steps               = 100,
    seed                        = SEED,
    fp16                        = torch.cuda.is_available(),
    report_to                   = 'none',
)

trainer_xlm_lora = Trainer(
    model           = xlm_lora,
    args            = args_xlm_lora,
    train_dataset   = xlm_train_ds,
    eval_dataset    = xlm_val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

print('Training XLM-RoBERTa LoRA...')
trainer_xlm_lora.train()
print('Done')

In [ ]:
plot_trainer_history(trainer_xlm_lora, 'XLM-RoBERTa LoRA (HF)')
f1_xlm_lora, acc_xlm_lora, preds_xlm_lora, _ = full_evaluation(
    trainer_xlm_lora, xlm_test_ds, 'XLM-RoBERTa LoRA (HF)'
)

## 4. MARBERT Experiments

In [ ]:
# ── MARBERT tokenizer and datasets ───────────────────────────────
print('Loading MARBERT tokenizer...')
mar_tok = AutoTokenizer.from_pretrained(MAR_MODEL)

print('Tokenizing datasets with MARBERT tokenizer...')
mar_train_ds = make_hf_dataset(train_df, mar_tok)
mar_val_ds   = make_hf_dataset(val_df,   mar_tok)
mar_test_ds  = make_hf_dataset(test_df,  mar_tok)

print(f'Train: {len(mar_train_ds):,} | Val: {len(mar_val_ds):,} | Test: {len(mar_test_ds):,}')

In [ ]:
# ── Strategy C: MARBERT Full Fine-Tuning ─────────────────────────
print('Loading MARBERT for Full Fine-Tuning...')
mar_full = AutoModelForSequenceClassification.from_pretrained(
    MAR_MODEL, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)

args_mar_full = TrainingArguments(
    output_dir                  = 'hf_outputs/marbert_full',
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate               = LR_FULL,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'macro_f1',
    greater_is_better           = True,
    logging_steps               = 100,
    seed                        = SEED,
    fp16                        = torch.cuda.is_available(),
    report_to                   = 'none',
)

trainer_mar_full = Trainer(
    model           = mar_full,
    args            = args_mar_full,
    train_dataset   = mar_train_ds,
    eval_dataset    = mar_val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

print('Training MARBERT Full Fine-Tuning...')
trainer_mar_full.train()
print('Done')

In [ ]:
plot_trainer_history(trainer_mar_full, 'MARBERT Full FT (HF)')
f1_mar_full, acc_mar_full, preds_mar_full, labels_mar_test = full_evaluation(
    trainer_mar_full, mar_test_ds, 'MARBERT Full FT (HF)'
)

In [ ]:
# ── Strategy D: MARBERT LoRA (peft library) ───────────────────────
print('Loading MARBERT for LoRA...')
mar_base_lora = AutoModelForSequenceClassification.from_pretrained(
    MAR_MODEL, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)

mar_lora = get_peft_model(mar_base_lora, lora_config)
mar_lora.print_trainable_parameters()

args_mar_lora = TrainingArguments(
    output_dir                  = 'hf_outputs/marbert_lora',
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate               = LR_LORA,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'macro_f1',
    greater_is_better           = True,
    logging_steps               = 100,
    seed                        = SEED,
    fp16                        = torch.cuda.is_available(),
    report_to                   = 'none',
)

trainer_mar_lora = Trainer(
    model           = mar_lora,
    args            = args_mar_lora,
    train_dataset   = mar_train_ds,
    eval_dataset    = mar_val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

print('Training MARBERT LoRA...')
trainer_mar_lora.train()
print('Done')

In [ ]:
plot_trainer_history(trainer_mar_lora, 'MARBERT LoRA (HF)')
f1_mar_lora, acc_mar_lora, preds_mar_lora, _ = full_evaluation(
    trainer_mar_lora, mar_test_ds, 'MARBERT LoRA (HF)'
)

## 5. Cross-Framework Comparison

In [ ]:
# ── HuggingFace Trainer results ───────────────────────────────────
hf_results = pd.DataFrame({
    'Model'         : ['XLM-RoBERTa', 'XLM-RoBERTa', 'MARBERT',    'MARBERT'],
    'Strategy'      : ['Full FT',      'LoRA',         'Full FT',    'LoRA'],
    'Framework'     : ['HF Trainer',   'HF Trainer',   'HF Trainer', 'HF Trainer'],
    'Test Macro F1' : [round(f1_xlm_full,  4), round(f1_xlm_lora,  4),
                       round(f1_mar_full,  4), round(f1_mar_lora,  4)],
    'Accuracy'      : [round(acc_xlm_full, 4), round(acc_xlm_lora, 4),
                       round(acc_mar_full, 4), round(acc_mar_lora, 4)],
})

# ── PyTorch results (from Notebook 01 — fill these in manually) ───
pytorch_results = pd.DataFrame({
    'Model'         : ['XLM-RoBERTa', 'XLM-RoBERTa', 'MARBERT',   'MARBERT'],
    'Strategy'      : ['Full FT',      'LoRA Scratch', 'Full FT',   'LoRA Scratch'],
    'Framework'     : ['PyTorch',      'PyTorch',      'PyTorch',   'PyTorch'],
    'Test Macro F1' : [0.8989, 0.8912, 0.9122, 0.9094],  # ← from Notebook 01
    'Accuracy'      : [0.96,   0.96,   0.97,   0.96  ],  # ← from Notebook 01
})

combined = pd.concat([pytorch_results, hf_results], ignore_index=True)
print('=== CROSS-FRAMEWORK COMPARISON ===')
print(combined.to_string(index=False))

combined.to_csv('hf_outputs/cross_framework_comparison.csv', index=False)
print('\nSaved: hf_outputs/cross_framework_comparison.csv')

In [ ]:
# ── Cross-framework comparison chart ─────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

pytorch_f1 = [0.8989, 0.8912, 0.9122, 0.9094]
hf_f1      = [f1_xlm_full, f1_xlm_lora, f1_mar_full, f1_mar_lora]
labels_exp = ['XLM\nFull FT', 'XLM\nLoRA', 'MARBERT\nFull FT', 'MARBERT\nLoRA']

x     = np.arange(len(labels_exp))
width = 0.35

bars1 = ax.bar(x - width/2, pytorch_f1, width, label='PyTorch (Notebook 01)', color='#4C72B0')
bars2 = ax.bar(x + width/2, hf_f1,      width, label='HF Trainer (Notebook 02)', color='#DD8452')

ax.bar_label(bars1, fmt='%.4f', padding=3, fontsize=9)
ax.bar_label(bars2, fmt='%.4f', padding=3, fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(labels_exp)
ax.set_ylabel('Test Macro F1')
ax.set_ylim(0.7, 1.0)
ax.set_title('Cross-Framework Comparison — PyTorch vs HuggingFace Trainer')
ax.legend()
plt.tight_layout()
plt.savefig('hf_outputs/cross_framework_chart.png', dpi=150)
plt.show()

print('\nKey finding: Both frameworks produce similar results.')
print('Difference in Macro F1 (PyTorch vs HF Trainer):')
for i, name in enumerate(labels_exp):
    diff = abs(pytorch_f1[i] - hf_f1[i])
    print(f'  {name.replace(chr(10)," ")}: {diff:.4f}')

In [ ]:
# ── Save everything to Google Drive ──────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import shutil
DRIVE_DIR = '/content/drive/MyDrive/sentimentgulf_checkpoints'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Save HF trainer checkpoints
for folder in ['hf_outputs/xlm_full', 'hf_outputs/xlm_lora',
               'hf_outputs/marbert_full', 'hf_outputs/marbert_lora']:
    # Find the best checkpoint inside each folder
    p = Path(folder)
    checkpoints = sorted(p.glob('checkpoint-*'), key=lambda x: int(x.name.split('-')[-1]))
    if checkpoints:
        best_ckpt = checkpoints[-1]
        dest = Path(DRIVE_DIR) / f'hf_{p.name}_best'
        if dest.exists(): shutil.rmtree(dest)
        shutil.copytree(best_ckpt, dest)
        print(f'✓ Saved: {best_ckpt.name} → {dest}')

# Save results CSV
shutil.copy('hf_outputs/cross_framework_comparison.csv',
            f'{DRIVE_DIR}/cross_framework_comparison.csv')
print('✓ Saved: cross_framework_comparison.csv')
print(f'\nAll saved to: {DRIVE_DIR}')